In [120]:
INPUT_DIR = "../outputs-step2/pairs/pair_04_05/summary.json"
OUTPUT_DIR = "../outputs-step3/pairs/pair_04_05/summary.json"


# Judge 改成「PDF 圖 + HTML 原始碼」對比，**只抓 formula error**：
# - HTML 以 text 形式進 user message（不是 image），PDF 仍為 image
# - 不再回 bbox（VLM 對 HTML text 沒有像素視角），改回 region_id（直接從 HTML data-region-id 取）
# - per-page 1 call（1 圖 + 1 段 HTML body），pair 內 sequential（避免 ollama multimodal 互踩）

SYSTEM_TEMPLATE = """你是學術論文 HTML 渲染品質的 *audit reviewer*，這次任務 **只審公式 (formula)**。

每次你會收到：
  - 1 張 image：原始 PDF 頁面（ground truth）
  - 1 段 HTML body 文字（render under test）：docling 對該頁產出，每個元素帶 `data-region-id="region-XXXX"`

你的工作：對比 PDF 與 HTML，**只報公式相關錯誤**，每個錯誤指到 HTML 裡那個元素的 `data-region-id`。

四種 formula 錯誤類型：

| type | 判定條件 |
|------|----------|
| formula_not_decoded   | HTML 該 region 出現 `<div class=\"formula-not-decoded\">Formula not decoded</div>` 之類 placeholder，但 PDF 對應位置實際有公式 |
| formula_as_text       | HTML 該 region 內公式被攤平成純文字（例如 `m t + 1 ∈ R n m`、`/Theta1`、希臘字母被拆成 `/beta`），沒有正確 MathML / LaTeX |
| formula_content_wrong | HTML 該 region 已有公式結構，但符號 / 上下標 / 整體結構與 PDF 明顯不符 |
| formula_missing       | PDF 該位置有公式，但 HTML 完全沒有對應 region（連 placeholder 都沒有） |

每個 error 必須帶 `region_id`，**只能用 HTML 文字裡實際出現過的 `data-region-id` 值**（例如 `region-0056`）。
`formula_missing` 例外：填鄰近文字 region 的 id，並在 description 講清楚是落在它前後。

**完全忽略**（不要報）：
- 非公式問題：純文字段落差異、表格欄位、圖片位置、reading order
- 字體、字距、antialiasing、padding
- 公式編號位置差異（編號文字存在即可，不論在右側 / 下方）

評分 (0-100 整數):
- 100: 該頁沒有任何公式錯誤
- 70-99: 1-2 個 low/medium formula error
- 30-69: 1 個 high (整段公式遺漏 / 全部沒解碼) 或多個 medium
- 0-29: 多個 high

若該頁本來就沒公式，且 HTML 也沒亂塞公式 → score=100, errors=[]。
"""

USER_TEMPLATE = """source page = {p}

【附件 image】= PDF page {p}（ground truth）

【HTML body (render under test)】
```html
{html_body}
```

請對比 PDF 與上述 HTML，**只抓 formula 相關錯誤**。

只輸出 JSON（不要 markdown code fence，不要前後文）：
{{
  "score": 0,
  "errors": [
    {{
      "type": "formula_not_decoded | formula_as_text | formula_content_wrong | formula_missing",
      "target_source_page": {p},
      "region_id": "region-XXXX",
      "description": "具體：哪個公式、PDF 顯示什麼、HTML 變成什麼。例如：'region-0056 對應 PDF 公式 (1) m_{{t+1}} = M(m_t, ε_{{t+1}})，HTML 顯示為 formula-not-decoded placeholder'",
      "severity": "high | medium | low"
    }}
  ]
}}

score 是 0-100 整數。region_id 必須出現在上面 HTML 文字裡（formula_missing 例外，挑鄰近 region）。
target_source_page 必為 {p}。
"""


In [121]:
from typing import Literal
from pydantic import BaseModel, Field

class JudgeError(BaseModel):
    type: Literal[
        "formula_not_decoded",
        "formula_as_text",
        "formula_content_wrong",
        "formula_missing",
    ]
    target_source_page: int = Field(description="此錯誤所在的原始 PDF 頁碼，必須 ∈ pair")
    region_id: str = Field(
        description="對應 HTML data-region-id (e.g. region-0056)；formula_missing 例外可用鄰近 region",
    )
    description: str
    severity: Literal["high", "medium", "low"]

class JudgeReport(BaseModel):
    pair: list[int]
    pair_slug: str
    score: float = Field(ge=0.0, le=100.0)  # 兩頁 score 取 min
    passed: bool
    pass_threshold: float
    errors: list[JudgeError]
    model_id: str
    raw_output: str  # 保留模型原始回應方便 debug

class JudgeLLMOutput(BaseModel):
    score: float = Field(ge=0.0, le=100.0)
    errors: list[JudgeError]


In [122]:
import json
import re
from pathlib import Path

# summary.json 內的路徑相對 project root（mkpdf/），notebook 在 notebooks/ 下執行
PROJECT_ROOT = Path("..")

_BODY_RE = re.compile(r"<body[^>]*>(.*?)</body>", re.DOTALL | re.IGNORECASE)


def extract_html_body(html_path: Path) -> str:
    """只送 <body> 內容給 VLM，drop <head>/<style> boilerplate（純樣式對 judge 沒資訊量）。"""
    html = html_path.read_text(encoding="utf-8")
    m = _BODY_RE.search(html)
    return m.group(1).strip() if m else html


def collect_pair_pages(pages: list[dict]) -> list[dict]:
    """把 pair summary.pages 攤平成 per-page judge 輸入。
    每筆含 source_page / pdf_path (image) / html_body (text)。
    page_pixel_size 不再需要（沒有 bbox 了，改用 region_id）。
    """
    result = []
    for page in pages:
        html_path = PROJECT_ROOT / page["rendered_html_path"]
        result.append({
            "source_page": page["source_page"],
            "pdf_path": PROJECT_ROOT / page["source_pdf_path"],
            "html_body": extract_html_body(html_path),
        })
    return result


In [123]:
import json
import re
import httpx
import ollama


PASS_THRESHOLD = 80.0  # 0-100 scale
JUDGE_TIMEOUT_SECONDS = 300
JUDGE_NUM_PREDICT = 16384  # thinking + JSON 預算


def make_pair_slug(p1: int, p2: int) -> str:
    return f"pair_{p1:02d}_{p2:02d}"


def _parse_judge_output(raw: str) -> JudgeLLMOutput | None:
    """先試直接 parse，失敗再 regex 抓最外層 {...}。"""
    try:
        return JudgeLLMOutput.model_validate_json(raw)
    except Exception:
        pass
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return None
    try:
        return JudgeLLMOutput.model_validate_json(m.group(0))
    except Exception:
        return None


def judge_single_page(
    page_info: dict,
    model_id: str,
    timeout_seconds: float = JUDGE_TIMEOUT_SECONDS,
    num_predict: int = JUDGE_NUM_PREDICT,
) -> tuple[JudgeLLMOutput | None, str, int]:
    """單頁：1 PDF image + 1 HTML body text → ollama → formula errors JSON。
    回傳 (parsed, raw_output, prompt_eval_count)。
    Timeout：parsed=None, raw="[TIMEOUT after Xs]", tokens=-1。

    Image drop 偵測：prompt_eval_count < 500 表示 PDF 圖沒吃進去。
    現在 prompt 主體是 HTML text + system，正常 ~3000-8000 tokens（HTML 多寡而定）。
    雷區（沿用前版實測）：
    - 不要 format='json'：跟 multimodal pipeline 不相容，image drop
    - 不要 num_ctx：reload model 走另一條 path，image drop
    """
    p = page_info["source_page"]
    usr_prompt = USER_TEMPLATE.format(p=p, html_body=page_info["html_body"])

    client = ollama.Client(timeout=timeout_seconds)
    try:
        response = client.chat(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_TEMPLATE},
                {
                    "role": "user",
                    "content": usr_prompt,
                    "images": [page_info["pdf_path"].read_bytes()],  # 只剩 PDF 是 image
                },
            ],
            think=True,
            options={"num_predict": num_predict},
        )
    except (httpx.TimeoutException, httpx.ReadTimeout, httpx.ConnectTimeout) as e:
        return None, f"[TIMEOUT after {timeout_seconds}s: {type(e).__name__}]", -1

    raw = response["message"]["content"]
    prompt_tokens = int(response.get("prompt_eval_count", -1))
    parsed = _parse_judge_output(raw)
    return parsed, raw, prompt_tokens


def judge_pair(
    input_path: str,
    output_path: str,
    model_id: str = "qwen3.5:9b",
    pass_threshold: float = PASS_THRESHOLD,
    timeout_seconds: float = JUDGE_TIMEOUT_SECONDS,
) -> tuple[JudgeReport, list[int]]:
    """對 pair 內每頁依序跑 judge，aggregate 後寫檔。"""
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    pages_info = collect_pair_pages(data["pages"])

    results = [judge_single_page(pi, model_id, timeout_seconds) for pi in pages_info]

    per_page_scores = []
    all_errors = []
    raw_chunks = []
    token_counts = []
    for pi, (parsed, raw, tokens) in zip(pages_info, results):
        token_counts.append(tokens)
        raw_chunks.append(
            f"=== source_page={pi['source_page']} prompt_eval_count={tokens} ===\n{raw}"
        )
        if parsed is None:
            per_page_scores.append(0.0)  # timeout / parse fail → 0
            continue
        per_page_scores.append(parsed.score)
        all_errors.extend(parsed.errors)

    score = min(per_page_scores) if per_page_scores else 0.0
    pair_pages = [pi["source_page"] for pi in pages_info]

    report = JudgeReport(
        pair=pair_pages,
        pair_slug=make_pair_slug(pair_pages[0], pair_pages[1]),
        score=score,
        passed=score >= pass_threshold,
        pass_threshold=pass_threshold,
        errors=all_errors,
        model_id=model_id,
        raw_output="\n\n".join(raw_chunks),
    )

    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(report.model_dump_json(indent=2), encoding="utf-8")
    return report, token_counts


In [124]:
report, token_counts = judge_pair(input_path=INPUT_DIR, output_path=OUTPUT_DIR)
print(f"pair={report.pair} score={report.score} passed={report.passed} formula_errors={len(report.errors)}")
print(f"per-page prompt_eval_count: {token_counts}  (<500 表示 PDF 圖被 drop；HTML body 越長 token 越多，~3000-8000 正常)")
print("--- errors ---")
for e in report.errors:
    print(f"  [{e.severity}] page={e.target_source_page} {e.region_id} type={e.type}")
    print(f"      {e.description}")
print("--- raw ---")
print(report.raw_output[:1500])


pair=[4, 5] score=0.0 passed=False formula_errors=0
per-page prompt_eval_count: [-1, -1]  (<500 表示 PDF 圖被 drop；HTML body 越長 token 越多，~3000-8000 正常)
--- errors ---
--- raw ---
=== source_page=4 prompt_eval_count=-1 ===
[TIMEOUT after 300s: ReadTimeout]

=== source_page=5 prompt_eval_count=-1 ===
[TIMEOUT after 300s: ReadTimeout]


In [125]:
# DIAGNOSTIC（已適配新版：PDF image + HTML text）
# 1. 資料本身（PDF 圖檔存在 & 大小 & 維度；HTML body 抽取後 char 數）
# 2. 最小 prompt + 同一張 PDF + HTML body → prompt_eval_count
# 3. 對比 wrapper（system + USER_TEMPLATE）→ prompt_eval_count
# 預期：2 OK 3 也 OK → 資料/wrapper 都沒問題
#       2 OK 3 爛  → SYSTEM/USER_TEMPLATE 觸發 ollama 走錯 path
#       2 也爛     → PDF 檔本身有問題

from PIL import Image

with open(INPUT_DIR) as f:
    pages_info = collect_pair_pages(json.load(f)["pages"])

for pi in pages_info:
    p = pi["source_page"]
    path = pi["pdf_path"]
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    dims = Image.open(path).size if exists else None
    print(f"page {p} PDF:  {path}  exists={exists}  bytes={size}  dims={dims}")
    print(f"page {p} HTML body chars: {len(pi['html_body'])}")

print()
print("--- Probe 2: minimal prompt + current data ---")
pi = pages_info[0]
r = ollama.chat(
    model="qwen3.5:9b",
    messages=[{
        "role": "user",
        "content": "Describe this image briefly.",
        "images": [pi["pdf_path"].read_bytes()],
    }],
    think=False,
)
print(f"prompt_eval_count={r['prompt_eval_count']}")
print(r["message"]["content"][:300])

print()
print("--- Probe 3: wrapper prompt (system + USER_TEMPLATE) + current data ---")
r = ollama.chat(
    model="qwen3.5:9b",
    messages=[
        {"role": "system", "content": SYSTEM_TEMPLATE},
        {
            "role": "user",
            "content": USER_TEMPLATE.format(p=pi["source_page"], html_body=pi["html_body"]),
            "images": [pi["pdf_path"].read_bytes()],
        },
    ],
    think=False,
)
print(f"prompt_eval_count={r['prompt_eval_count']}")
print(r["message"]["content"][:500])


page 4 PDF:  ../outputs-step2/pages/page_04/source_pdf.png  exists=True  bytes=268265  dims=(1089, 1486)
page 4 HTML body chars: 6907
page 5 PDF:  ../outputs-step2/pages/page_05/source_pdf.png  exists=True  bytes=341825  dims=(1089, 1486)
page 5 HTML body chars: 7340

--- Probe 2: minimal prompt + current data ---


KeyboardInterrupt: 

In [ ]:
# r = ollama.chat(
#     model="qwen3.5:9b",
#     messages=[
#         {"role": "system", "content": SYSTEM_TEMPLATE},
#         {
#             "role": "user",
#             "content": "Describe these 2 images briefly.",
#             "images": [
#                 Path("../outputs-step2/pages/page_01/source_pdf.png").read_bytes(),
#                 Path("../outputs-step2/pages/page_01/rendered_screenshot.png").read_bytes(),
#             ],
#         },
#     ],
#     think=False,
# )
# print(r["prompt_eval_count"], "|", r["message"]["content"][:300])

In [ ]:
# import ollama

# r = ollama.chat(
#     model="qwen3.5:9b",
#     messages=[
#         {"role": "system", "content": SYSTEM_TEMPLATE},
#         {
#             "role": "user",
#             "content": "What do you see in this image? if info is not enough, just say 'not enough info'.",
#             "images": ["../outputs-step2/pages/page_01/source_pdf.png"],
#         },
#     ],
# )
# print(r["prompt_eval_count"], "|", r["message"]["content"][:200])

In [ ]:
# r = ollama.chat(
#     model="qwen3.5:9b",
#     messages=[{
#         "role": "user",
#         "content": "Describe these 4 images briefly.",
#         "images": [
#             "../outputs-step2/pages/page_01/source_pdf.png",
#             "../outputs-step2/pages/page_01/rendered_screenshot.png",
#             "../outputs-step2/pages/page_02/source_pdf.png",
#             "../outputs-step2/pages/page_02/rendered_screenshot.png",
#         ],
#     }]
# )
# print(r["prompt_eval_count"], "|", r["message"]["content"][:200])